# Preflight - does every configuration run at all

**Run this kernel first.** It exercises every architecture, head and feature-set
combination at `--quick` (1 seed, 2 origins, 25 epochs) and reports which ones
construct, train and score without error.

The numbers it produces are **degraded and must never be reported**. The only
output that matters is the PASS/FAIL table at the end. A wide multivariate input
changes the input width every architecture is built with -- 3 channels becomes
51 -- and that is exactly the kind of thing that fails at construction time,
after a full kernel has already spent hours on the runs before it.

## 1. Environment

In [ ]:
import subprocess, sys, os, json, time, hashlib, re
from pathlib import Path

REPO = "https://github.com/MLOpenSourceOpenScience/disease_modeling_MLOS2.git"
COMMIT = "45f1c0878f002407633ed1237638734faa9ceb2b"
BLOB_NPY = "f7cfa6ec31a4058584fe256a1d6de6800e72a5b1"
BLOB_ADJ = "f3a3cb7f43998850410b0a494f16f331c3830a84"
SEGMENTS = [0.6, 0.7, 0.8, 0.9, 1.0]     # the authors' __main__ segment list

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
# Build outside /kaggle/working: anything left there becomes kernel output, and a
# 40 MB clone plus a venv makes `kaggle kernels output` unusably slow.
SCRATCH = Path("/tmp/repro") if Path("/tmp").exists() else WORK
SCRATCH.mkdir(parents=True, exist_ok=True)
SRC = SCRATCH / "mlos2"
VENV = SCRATCH / "venv311"
PY311 = VENV / "bin" / "python"

os.environ["MPLBACKEND"] = "Agg"          # their plotting helpers call plt.show()


def sh(*args, check=True, quiet=False, **kw):
    """Run a command, echoing it, and abort on a non-zero exit."""
    if not quiet:
        print("$", " ".join(str(a) for a in args))
    r = subprocess.run([str(a) for a in args], text=True, capture_output=True, **kw)
    if r.stdout.strip() and not quiet:
        print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-4000:])
        if check:
            raise SystemExit("command failed: " + " ".join(str(a) for a in args))
    return r


print("kernel python:", sys.version.split()[0])

In [ ]:
# Kaggle runs Python 3.12; torch 2.1.2 has no cp312 wheel. Fetch a standalone
# 3.11 with uv rather than bumping the authors' pinned torch.
sh(sys.executable, "-m", "pip", "install", "-q", "uv")

UV = [sys.executable, "-m", "uv"]
sh(*UV, "python", "install", "3.11")
sh(*UV, "venv", "--python", "3.11", str(VENV))

PIP = [*UV, "pip", "install", "-q", "--python", str(PY311)]

# Exactly the versions in the authors' requirements.txt.
sh(*PIP, "torch==2.1.2", "--index-url", "https://download.pytorch.org/whl/cpu")
sh(*PIP, "torch_scatter", "torch_sparse", "-f",
   "https://data.pyg.org/whl/torch-2.1.2+cpu.html")

# Deviation 2: the authors pin torch_geometric==2.5.3, but PGT 0.54.0 imports
# torch_geometric.utils.to_dense_adj, which PyG removed in 2.4. 2.4.0 is the
# newest release where all five architectures import.
sh(*PIP, "torch_geometric==2.4.0", "numpy~=1.26.2", "pandas~=2.2.0",
   "scikit_learn==1.4.0", "statsmodels==0.14.1", "decorator==4.4.2",
   "cython", "matplotlib", "tqdm")

# Deviation 1: PGT's own pandas<=1.3.5 pin contradicts the authors'
# pandas~=2.2.0 and has no Python 3.11 wheel.
sh(*PIP, "--no-deps", "torch_geometric_temporal==0.54.0")

In [ ]:
probe = sh(str(PY311), "-c", """
import json, torch, torch_geometric, pandas, numpy
from torch_geometric_temporal import A3TGCN, ASTGCN, AAGCN
from torch_geometric_temporal.nn.recurrent import DCRNN
from torch_geometric_temporal.signal import StaticGraphTemporalSignal, temporal_signal_split
print(json.dumps({
    "python": ".".join(map(str, __import__("sys").version_info[:3])),
    "torch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "pandas": pandas.__version__,
    "numpy": numpy.__version__,
}))
""", quiet=True)

versions = json.loads(probe.stdout.strip().splitlines()[-1])
print(json.dumps(versions, indent=2))
assert versions["python"].startswith("3.11"), versions["python"]
assert versions["torch"].startswith("2.1.2"), versions["torch"]
assert versions["torch_geometric"] == "2.4.0", versions["torch_geometric"]
print("all five architectures import OK under Python 3.11")

## 2. Clone

In [ ]:
PROJECT = "https://github.com/rathishTharusha/dengue-forecasting-gnn.git"
BRANCH = "exp/beat-baseline"
PROJ = SCRATCH / "project"

if not PROJ.exists():
    sh("git", "clone", "--depth", "1", "--branch", BRANCH, PROJECT, str(PROJ))
print("Cloned branch:", BRANCH)

RUNNER = PROJ / "analysis" / "_build" / "run_beat_baseline.py"
assert RUNNER.exists(), f"Runner not found at {RUNNER}"

## 3. Smoke every configuration

In [ ]:
COMBOS = [
    ("A3TGCN", "det", "cases"),
    ("A3TGCN", "gauss", "cases"),
    ("A3TGCN", "nb", "cases"),
    ("A3TGCN", "det", "causal"),
    ("A3TGCN", "det", "climate"),
    ("STGAT", "det", "climate"),
    ("STGAT", "nb", "causal"),
    ("ASTGCN", "det", "climate"),
    ("AAGCN", "det", "causal"),
    ("DCRNN", "det", "cases"),
]

results = []
for arch_name, head, fset in COMBOS:
    label = f"{arch_name}/{head}/{fset}"
    started = time.time()
    proc = subprocess.run(
        [str(PY311), "-u", str(RUNNER), "--arch", arch_name, "--head", head,
         "--features", fset, "--quick", "--out-dir", str(SCRATCH / "preflight"),
         "--tag", f"pre_{arch_name}_{head}_{fset}"],
        cwd=str(PROJ), capture_output=True, text=True,
    )
    ok = proc.returncode == 0
    tail = "" if ok else (proc.stdout + proc.stderr).strip().splitlines()[-1][:200]
    results.append((label, ok, time.time() - started, tail))
    print(f"{'PASS' if ok else 'FAIL'}  {label:28s} {time.time() - started:6.1f}s  {tail}",
          flush=True)

print()
n_ok = sum(1 for _, ok, _, _ in results if ok)
print(f"{n_ok}/{len(results)} configurations run.")
for label, ok, _, tail in results:
    if not ok:
        print(f"  FAILED {label}: {tail}")
assert n_ok == len(results), "fix the failures above before launching the full sweeps"
print("All configurations run. Safe to launch the full kernels.")